# 05 - 影像科多模态 LoRA 微调（Qwen2-VL-2B-Instruct）⭐

**这是论文最大的亮点**：训练一个能看图回答问题的影像科 Agent。

**模型**：`Qwen/Qwen2-VL-2B-Instruct`（2B，本地友好）

**显存**：10-12 GB（5070 Ti 笔记本勉强够，建议关掉其他程序）

**时间**：3-4 小时

**数据**：
- 多模态：HuggingFace 上的 SLAKE 中英文医学 VQA 数据集（自动下载）
- 文本：本地的 `radiologist_text_train.jsonl`

**升级**：云端可换 `Qwen/Qwen2.5-VL-3B-Instruct` 或 `Qwen/Qwen2.5-VL-7B-Instruct`

## Step 1: 环境检查

In [ ]:
import torch, transformers, peft
from pathlib import Path

print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# 检查必要库
try:
    from qwen_vl_utils import process_vision_info
    print('qwen-vl-utils 已安装 ✅')
except ImportError:
    print('需要安装：pip install qwen-vl-utils')

## Step 2: 配置

In [ ]:
MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'
# 云端可换：
# MODEL_NAME = 'Qwen/Qwen2.5-VL-3B-Instruct'  # 24GB GPU
# MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'  # A100 40GB

OUTPUT_DIR = Path('../models/radiologist_vl_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 显存敏感参数
MAX_LENGTH = 768
BATCH_SIZE = 1            # 多模态非常吃显存，只能 1
GRAD_ACCUM = 8            # 实际 batch = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 2

LORA_R = 8
LORA_ALPHA = 16
USE_4BIT = True           # 多模态推荐开 4-bit 量化省显存

# 数据集（HuggingFace 自动下载）
VQA_DATASET = 'BoKelvin/SLAKE'  # 中英文医学 VQA
MAX_TRAIN_SAMPLES = 2000        # 不要全部，控制训练时间

print(f'模型: {MODEL_NAME}')
print(f'输出: {OUTPUT_DIR}')

## Step 3: 加载多模态数据集（SLAKE）

SLAKE 数据集结构：每条包含 image、question、answer。

In [ ]:
from datasets import load_dataset

# 第一次会下载约 1-2 GB
print('下载 SLAKE 数据集（首次需要几分钟）...')
ds = load_dataset(VQA_DATASET)
print(f'数据集结构: {ds}')
print(f'\n样例:')
sample = ds['train'][0]
for k, v in sample.items():
    if hasattr(v, 'size'):  # 图片对象
        print(f'  {k}: <Image {v.size}>')
    else:
        print(f'  {k}: {str(v)[:100]}')

In [ ]:
# 转换为 Qwen-VL 训练格式
# Qwen-VL 期望的 message 格式：
# [
#   {'role': 'user', 'content': [{'type': 'image', 'image': PIL.Image}, {'type': 'text', 'text': '...'}]},
#   {'role': 'assistant', 'content': [{'type': 'text', 'text': '...'}]}
# ]

def format_sample(sample):
    # SLAKE 字段名可能不同，做兼容处理
    image = sample.get('image') or sample.get('img') or sample.get('Image')
    question = sample.get('question') or sample.get('Question') or ''
    answer = sample.get('answer') or sample.get('Answer') or ''
    
    # 中文 / 英文都保留
    return {
        'image': image,
        'question': str(question),
        'answer': str(answer),
    }

import random
random.seed(42)

all_samples = [format_sample(s) for s in ds['train']]
all_samples = [s for s in all_samples if s['image'] is not None and s['question'] and s['answer']]
random.shuffle(all_samples)
all_samples = all_samples[:MAX_TRAIN_SAMPLES]

n_val = max(50, int(len(all_samples) * 0.05))
train_samples = all_samples[n_val:]
val_samples = all_samples[:n_val]

print(f'训练样本: {len(train_samples)}, 验证: {len(val_samples)}')

## Step 4: 加载基座模型

In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

model_kwargs = {
    'torch_dtype': torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    'device_map': 'auto',
    'trust_remote_code': True,
}

if USE_4BIT:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForVision2Seq.from_pretrained(MODEL_NAME, **model_kwargs)
model.config.use_cache = False
print(f'模型加载完成: {MODEL_NAME}')

## Step 5: LoRA 配置（只训练语言模型部分，视觉编码器冻结）

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6: 数据预处理 + DataLoader

In [ ]:
from torch.utils.data import Dataset

SYSTEM_PROMPT = '你是一名影像科医生，根据医学影像和问题给出专业、循证的回答。'

class VQADataset(Dataset):
    def __init__(self, samples, processor, max_length=768):
        self.samples = samples
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': s['image']},
                {'type': 'text', 'text': s['question']},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': s['answer']}]},
        ]
        
        # 应用 chat template
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        
        # 处理图片
        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt',
        )
        
        # 把 labels 设为 input_ids（对所有 token 都计算 loss，简单起见）
        inputs['labels'] = inputs['input_ids'].clone()
        
        # 去掉 batch 维
        return {k: v.squeeze(0) if hasattr(v, 'squeeze') else v for k, v in inputs.items()}

train_ds = VQADataset(train_samples, processor, MAX_LENGTH)
val_ds = VQADataset(val_samples, processor, MAX_LENGTH)

# 测试一条数据
sample0 = train_ds[0]
for k, v in sample0.items():
    if hasattr(v, 'shape'):
        print(f'{k}: {v.shape}, {v.dtype}')

## Step 7: 训练（自定义 collator）

In [ ]:
import os
os.environ['WANDB_DISABLED'] = 'true'

from transformers import TrainingArguments, Trainer

def collate_fn(batch):
    # batch 是 [{...}, {...}]，需要 padding 到统一长度
    # 简单实现：直接 stack（适用 batch_size=1）
    if len(batch) == 1:
        return {k: v.unsqueeze(0) if hasattr(v, 'unsqueeze') else [v] for k, v in batch[0].items()}
    
    # batch_size > 1 时手动 padding（一般 VL 模型用 batch_size=1）
    raise NotImplementedError('VL 训练建议 batch_size=1，使用 gradient_accumulation 来增大有效 batch')

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=20,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none',
    seed=42,
    remove_unused_columns=False,  # 重要！VL 模型有特殊字段
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate_fn,
)

print('开始多模态 LoRA 训练（耗时较长，建议留足时间）...')
trainer.train()

## Step 8: 保存

In [ ]:
import json

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

with open(OUTPUT_DIR / 'training_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'base_model': MODEL_NAME,
        'role': 'radiologist',
        'modality': 'vision-language',
        'dataset': VQA_DATASET,
        'train_size': len(train_ds),
        'val_size': len(val_ds),
        'epochs': NUM_EPOCHS,
        'use_4bit': USE_4BIT,
    }, f, ensure_ascii=False, indent=2)

print(f'多模态 LoRA 已保存: {OUTPUT_DIR.resolve()}')

## Step 9: 推理测试（用验证集的图）

In [ ]:
from qwen_vl_utils import process_vision_info

model.config.use_cache = True
model.eval()

for s in val_samples[:3]:
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': s['image']},
            {'type': 'text', 'text': s['question']},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    response = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    
    print(f'\n问: {s["question"]}')
    print(f'标准答案: {s["answer"]}')
    print(f'模型回答: {response}')
    print('-' * 80)